# OMI · Dal file grezzo alla tabella pulita

Questo notebook trasforma i 20 file semestrali delle *Quotazioni immobiliari* dell'OMI (Agenzia delle Entrate, 2016-2025) in un'unica tabella pulita.

- **Input:** `data/Raw/Omi/*_VALORI.csv`, i file originali, che non vengono mai modificati.
- **Output:** `data/processed/omi_semestri_puliti.csv`, una riga per zona OMI × tipologia × semestre, solo abitazioni, negozi e uffici.

| Fase | Contenuto |
|---|---|
| 1 | Scheda della fonte e dizionario delle colonne |
| 2-3 | Lettura e struttura del file |
| 4 | Controlli di qualità su un semestre |
| 5 | Decisioni di pulizia (un semestre) |
| 6 | Tutti i semestri e controlli sul totale |
| 7 | Nomi, ripartizione geografica, date |
| 8 | Salvataggio, decisioni, limiti dei dati |



In [1]:
import pandas as pd
import os
import re

CARTELLA_OMI = "../data/Raw/Omi"
CARTELLA_OUT = "../data/processed"

PREZZI = ["Compr_min", "Compr_max", "Loc_min", "Loc_max"]
pd.set_option("display.max_columns", 30)

# Fase 1 · Scheda della fonte

| Voce | Valore |
|---|---|
| Ente | Agenzia delle Entrate, Osservatorio del Mercato Immobiliare (OMI) |
| Servizio | Forniture dati OMI, quotazioni immobiliari |
| Licenza | CC-BY 4.0 |
| Come citare | «Agenzia delle Entrate – OMI» |
| Periodo | 2016-2025, semestrale (20 semestri, da 2016-H1 a 2025-H2) |
| Elaborazione dei file | 6 ottobre 2026 (prima riga di ogni file) |
| Unità | Acquisto: €/m². Affitto: €/m² **al mese** |
| Una riga è | una zona OMI × una tipologia × uno stato di conservazione (vedi 4.2 e 4.7) |

L'unità mensile dell'affitto è verificata in 6.6.

**Limiti dichiarati dall'ente**
- Le quotazioni sono intervalli minimo/massimo stimati, «di larga massima», non prezzi di vendita.
- L'elenco dei comuni può cambiare da un semestre all'altro.
- Per negozi e centri commerciali il giudizio O/N/S riguarda la posizione commerciale, non lo stato dell'immobile.

## Dizionario delle colonne

| Colonna | Significato | Verifica |
|---|---|---|
| `Area_territoriale` | Ripartizione OMI: Nord-Ovest, Nord-Est, Centro, Sud, Isole | 7.2 |
| `Regione`, `Prov`, `Comune_ISTAT`, `Comune_descrizione` | Posizione amministrativa (`Prov` contiene «NA» di Napoli) | 4.7 |
| `Comune_cat`, `Sez`, `Comune_amm` | Codici catastali e amministrativi, non usati | escluse in lettura |
| `Fascia` | Posizione della zona: B centrale, C semicentrale, D periferica, E suburbana, R rurale | 4.6 |
| `Zona` | Codice della zona nel comune (es. B1); da solo non è unico | 4.3 |
| `LinkZona` | Identificativo unico della zona | 4.3 |
| `Cod_Tip`, `Descr_Tipologia` | Codice e descrizione della tipologia (18 valori) | 3 e 4.3 |
| `Stato` | Stato di conservazione: OTTIMO, NORMALE, SCADENTE | 3 e 4.7 |
| `Stato_prev` | «P» nella riga dello stato prevalente; vuoto nelle altre righe della stessa zona e tipologia | 4.2 e 4.7 |
| `Compr_min`, `Compr_max` | Intervallo del prezzo di acquisto, €/m² | 4.4 e 4.5 |
| `Loc_min`, `Loc_max` | Intervallo dell'affitto, €/m² al mese | 4.4, 4.5, 6.6 |
| `Sup_NL_compr`, `Sup_NL_loc` | Superficie di riferimento del €/m²: **L** lorda (con i muri), **N** netta | 3 e 4.7 |

**Superficie.** Il prezzo di acquisto è sempre su superficie lorda; l'affitto lo è in circa tre righe su quattro, nelle altre è netta (4.7). Per questo `Sup_NL_loc` resta nella tabella pulita: chi la usa può distinguere i due casi.

**Stato prevalente.** Per la stessa zona e tipologia l'OMI può pubblicare più righe, una per stato di conservazione. `Stato_prev` = «P» segna la quotazione principale, quella che si tiene (Decisione 4).

# Fase 2 · Leggere il file

Ogni semestre ha due file: `..._VALORI.csv` (prezzi) e `..._ZONE.csv` (descrizione delle zone). Si usano solo i VALORI.

In [2]:
file_valori = sorted([f for f in os.listdir(CARTELLA_OMI) if f.endswith("_VALORI.csv")])
print(len(file_valori), "file VALORI")
file_valori[:3]

20 file VALORI


['QI_1444537_1_20161_VALORI.csv',
 'QI_1444538_1_20162_VALORI.csv',
 'QI_1444540_1_20171_VALORI.csv']

Prima di usare pandas si guardano le righe grezze del primo file, per capire separatore, righe da saltare e decimali.

In [3]:
percorso = os.path.join(CARTELLA_OMI, file_valori[0])
with open(percorso, encoding="utf-8") as f:
    for i in range(4):
        print(f.readline())

Quotazioni Immobiliari : Valori di Mercato - Semestre 2016/1 - elaborazione del 06-OTT-26

Area_territoriale;Regione;Prov;Comune_ISTAT;Comune_cat;Sez;Comune_amm;Comune_descrizione;Fascia;Zona;LinkZona;Cod_Tip;Descr_Tipologia;Stato;Stato_prev;Compr_min;Compr_max;Sup_NL_compr;Loc_min;Loc_max;Sup_NL_loc;

NORD-OVEST;PIEMONTE;AL;1006003;A2AA; ;A182;ALESSANDRIA;B;B1;AL00000001;20;Abitazioni civili;NORMALE;P;770;1050;L;3,1;4,2;L;

NORD-OVEST;PIEMONTE;AL;1006003;A2AA; ;A182;ALESSANDRIA;B;B1;AL00000001;13;Box;NORMALE;P;1100;1600;L;4,4;6,2;L;



Cosa serve per leggerlo:
- La riga 1 è un titolo, non fa parte della tabella → `skiprows=1`.
- Il separatore è `;` → `sep=";"`.
- I decimali usano la virgola → `decimal=","`.
- `NA` è la sigla di Napoli, ma pandas la scambia per dato mancante → `keep_default_na=False` e `na_values=[""]`.
- `Comune_ISTAT` è un codice, quindi si legge come testo.

In [4]:
df = pd.read_csv(percorso, sep=";", skiprows=1, decimal=",",
                 keep_default_na=False, na_values=[""],
                 dtype={"Comune_ISTAT": str})
df.shape

(162193, 22)

Il primo semestre (2016-H1) ha 162.193 righe e 22 colonne, una in più delle 21 attese (vedi Fase 3).

# Fase 3 · Descrivere la struttura

In [5]:
df.head()

,Area_territoriale,Regione,Prov,Comune_ISTAT,Comune_cat,Sez,Comune_amm,Comune_descrizione,Fascia,Zona,LinkZona,Cod_Tip,Descr_Tipologia,Stato,Stato_prev,Compr_min,Compr_max,Sup_NL_compr,Loc_min,Loc_max,Sup_NL_loc,Unnamed: 21
0,NORD-OVEST,PIEMONTE,AL,1006003,A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,20,Abitazioni civili,NORMALE,P,770.0,1050.0,L,3.1,4.2,L,NaN
1,NORD-OVEST,PIEMONTE,AL,1006003,A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,13,Box,NORMALE,P,1100.0,1600.0,L,4.4,6.2,L,NaN
2,NORD-OVEST,PIEMONTE,AL,1006003,A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,14,Posti auto coperti,NORMALE,P,700.0,1000.0,L,2.7,3.8,L,NaN
3,NORD-OVEST,PIEMONTE,AL,1006003,A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,15,Posti auto scoperti,NORMALE,P,600.0,800.0,L,2.5,3.3,L,NaN
4,NORD-OVEST,PIEMONTE,AL,1006003,A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,9,Magazzini,NORMALE,P,900.0,1300.0,L,4.8,5.7,L,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162193 entries, 0 to 162192
Data columns (total 22 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Area_territoriale   162193 non-null  object 
 1   Regione             162193 non-null  object 
 2   Prov                162193 non-null  object 
 3   Comune_ISTAT        162192 non-null  object 
 4   Comune_cat          162193 non-null  object 
 5   Sez                 162193 non-null  object 
 6   Comune_amm          162193 non-null  object 
 7   Comune_descrizione  162193 non-null  object 
 8   Fascia              162193 non-null  object 
 9   Zona                162193 non-null  object 
 10  LinkZona            162193 non-null  object 
 11  Cod_Tip             162193 non-null  int64  
 12  Descr_Tipologia     162193 non-null  object 
 13  Stato               162181 non-null  object 
 14  Stato_prev          162193 non-null  object 
 15  Compr_min           162180 non-nul

- `decimal=","` ha funzionato: i prezzi sono numeri.
- `Unnamed: 21` è vuota: nasce dal `;` alla fine di ogni riga e si elimina.
- L'affitto manca in circa l'8,5% delle righe, il prezzo di acquisto quasi mai.

In [7]:
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df.shape

(162193, 21)

Valori distinti delle colonne di categoria (`dropna=False` mostra anche i vuoti).

In [8]:
for colonna in ["Area_territoriale", "Fascia", "Stato", "Stato_prev", "Sup_NL_compr", "Sup_NL_loc"]:
    print(df[colonna].value_counts(dropna=False))
    print()

Area_territoriale
NORD-OVEST    50367
SUD           40384
NORD-EST      31959
CENTRO        26474
ISOLE         13009
Name: count, dtype: int64

Fascia
B    67172
D    31686
R    27390
E    25039
C    10906
Name: count, dtype: int64

Stato
NORMALE     141869
OTTIMO       19726
SCADENTE       586
NaN             12
Name: count, dtype: int64

Stato_prev
P    148787
      13406
Name: count, dtype: int64

Sup_NL_compr
L      162180
NaN        13
Name: count, dtype: int64

Sup_NL_loc
L      111746
N       36444
NaN     13607
          396
Name: count, dtype: int64



- `Area_territoriale` e `Fascia` hanno 5 valori ciascuna.
- `Stato`: NORMALE 141.869 righe, OTTIMO 19.726, SCADENTE 586, 12 senza valore.
- `Stato_prev`: «P» in 148.787 righe, «vuoto» in 13.406. Quel vuoto è una cella con soli spazi, non un dato mancante per pandas: si ripulisce in 6.7.
- `Sup_NL_compr` è sempre L (13 righe senza valore). `Sup_NL_loc` è L in 111.746 righe e N in 36.444; 13.607 sono mancanti e 396 hanno solo uno spazio.

Le tipologie con il loro codice:

In [9]:
df[["Cod_Tip", "Descr_Tipologia"]].drop_duplicates().sort_values("Cod_Tip")

,Cod_Tip,Descr_Tipologia
12,1,Ville e Villini
5,5,Negozi
6,6,Uffici
17,7,Capannoni tipici
16,8,Capannoni industriali
4,9,Magazzini
7,10,Laboratori
1,13,Box
2,14,Posti auto coperti
3,15,Posti auto scoperti


Sono 18 tipologie. I codici non sono consecutivi, e non serve che lo siano.

# Fase 4 · Controlli di qualità

I controlli sono fatti sul primo semestre (2016-H1, tutte le tipologie) e servono a decidere come pulire. La Fase 6 li ripete sul totale.

## 4.1 Valori mancanti

In [10]:
mancanti = df.isna().sum()
percentuale = (mancanti / len(df) * 100).round(2)
pd.DataFrame({"mancanti": mancanti, "percentuale": percentuale})[mancanti > 0]

,mancanti,percentuale
Comune_ISTAT,1,0.00
Stato,12,0.01
Compr_min,13,0.01
Compr_max,13,0.01
Sup_NL_compr,13,0.01
Loc_min,13811,8.52
Loc_max,13811,8.52
Sup_NL_loc,13607,8.39


Solo l'affitto e `Sup_NL_loc` hanno molti vuoti (8,4-8,5%): non per tutte le zone esiste una quotazione di affitto. I prezzi di acquisto mancano in 13 righe su 162.193.

## 4.2 Duplicati e chiave

La chiave dovrebbe essere zona + tipologia. Se la stessa coppia compare più volte, di solito cambia lo stato di conservazione.

In [11]:
CHIAVE = ["LinkZona", "Cod_Tip"]

print("Righe identiche:", df.duplicated().sum())
print("Righe con chiave ripetuta:", df.duplicated(subset=CHIAVE).sum())
print("Zone distinte:", df["LinkZona"].nunique())

Righe identiche: 0
Righe con chiave ripetuta: 13406
Zone distinte: 24339


In [12]:
ripetute = df[df.duplicated(subset=CHIAVE, keep=False)].sort_values(CHIAVE)
ripetute[CHIAVE + ["Stato", "Stato_prev", "Compr_min", "Compr_max"]].head(10)

,LinkZona,Cod_Tip,Stato,Stato_prev,Compr_min,Compr_max
94168,AN00000034,20,NORMALE,P,1300.0,1800.0
94169,AN00000034,20,OTTIMO,,1700.0,2050.0
94190,AN00000037,20,NORMALE,P,1400.0,1800.0
94191,AN00000037,20,OTTIMO,,1650.0,2300.0
94134,AN00000038,5,SCADENTE,,1500.0,2100.0
94135,AN00000038,5,OTTIMO,,4600.0,7200.0
94136,AN00000038,5,NORMALE,P,2950.0,4600.0
94394,AN00000126,20,NORMALE,P,1100.0,1550.0
94395,AN00000126,20,OTTIMO,,1550.0,1900.0
94713,AN00000235,5,OTTIMO,,2950.0,5300.0


Non ci sono righe identiche, ma 13.406 righe ripetono una coppia zona + tipologia già presente. Cambiano `Stato` e `Stato_prev`: c'è una riga per ogni stato di conservazione (per i negozi di `AN00000038`, SCADENTE, OTTIMO e NORMALE con prezzi molto diversi). La chiave è quindi zona + tipologia + stato. Il numero 13.406 ricompare in 4.7.

## 4.3 Significato dei codici

Entrambi i numeri devono essere 1: un codice tipologia ha una sola descrizione, un codice zona sta in un solo comune.

In [13]:
print(df.groupby("Cod_Tip")["Descr_Tipologia"].nunique().max())
print(df.groupby("LinkZona")["Comune_ISTAT"].nunique().max())

1
1


Entrambi valgono 1: `Cod_Tip` e `LinkZona` sono affidabili.

## 4.4 Valori impossibili

In [14]:
print("Acquisto con minimo > massimo:", (df["Compr_min"] > df["Compr_max"]).sum())
print("Affitto con minimo > massimo:", (df["Loc_min"] > df["Loc_max"]).sum())
print("Prezzi negativi:", (df[PREZZI] < 0).sum().sum())
print()
print("Prezzi uguali a zero:")
print((df[PREZZI] == 0).sum())

Acquisto con minimo > massimo: 0
Affitto con minimo > massimo: 0
Prezzi negativi: 0

Prezzi uguali a zero:
Compr_min    215
Compr_max    215
Loc_min      373
Loc_max      373
dtype: int64


In [15]:
print(df[df["Compr_min"] == 0]["Descr_Tipologia"].value_counts())
print()
print(df[df["Loc_min"] == 0]["Descr_Tipologia"].value_counts())

Descr_Tipologia
Abitazioni civili               47
Capannoni tipici                18
Magazzini                       17
Negozi                          17
Uffici                          17
Laboratori                      17
Abitazioni di tipo economico    16
Box                             16
Posti auto coperti              16
Posti auto scoperti             16
Ville e Villini                 15
Autorimesse                      3
Name: count, dtype: int64

Descr_Tipologia
Abitazioni civili               52
Abitazioni di tipo economico    46
Laboratori                      44
Negozi                          42
Box                             41
Ville e Villini                 41
Uffici                          39
Magazzini                       31
Capannoni industriali           14
Capannoni tipici                14
Posti auto scoperti              5
Posti auto coperti               4
Name: count, dtype: int64


Nessun prezzo negativo e nessun minimo maggiore del massimo. Ci sono prezzi uguali a zero (215 per l'acquisto, 373 per l'affitto) su quasi tutte le tipologie: non sono prezzi credibili, sono dati mancanti (Decisione 2).

## 4.5 Valori anomali

In [16]:
df[PREZZI].describe()

,Compr_min,Compr_max,Loc_min,Loc_max
count,162180.000000,162180.000000,148382.000000,148382.000000
mean,805.104945,1089.866414,3.145975,4.347260
std,606.640883,836.505220,2.249013,3.187223
min,0.000000,0.000000,0.000000,0.000000
25%,400.000000,570.000000,1.700000,2.400000
50%,650.000000,890.000000,2.600000,3.500000
75%,1000.000000,1350.000000,4.000000,5.400000
max,16300.000000,25000.000000,79.000000,120.000000


In [17]:
colonne_vista = ["Comune_descrizione", "Descr_Tipologia", "Loc_max"]
df.sort_values("Loc_max", ascending=False).head(5)[colonne_vista]

,Comune_descrizione,Descr_Tipologia,Loc_max
31482,MILANO,Negozi,120.0
81901,RICCIONE,Negozi,90.0
57013,CORTINA D`AMPEZZO,Negozi,64.0
105548,ROMA,Negozi,63.0
105555,ROMA,Negozi,59.3


In [18]:
df.groupby("Descr_Tipologia")[["Compr_max", "Loc_max"]].max().sort_values("Loc_max", ascending=False)

,Compr_max,Loc_max
Descr_Tipologia,,
Negozi,25000.0,120.0
Ville e Villini,18900.0,54.7
Abitazioni signorili,22900.0,49.0
Abitazioni civili,15900.0,42.0
Uffici,10200.0,40.2
Uffici strutturati,9100.0,40.0
Centri commerciali,7100.0,33.8
Abitazioni di tipo economico,9600.0,30.0
Box,7200.0,26.6


I massimi di affitto sono negozi nelle località più care (Milano 120 €/m² al mese, Riccione, Cortina d'Ampezzo, Roma); tra le abitazioni sono signorili e ville. Il massimo assoluto, un negozio a Milano, è molto sopra gli altri ma plausibile per quel mercato. Sono estremi veri, non errori.

## 4.6 Coerenza tra colonne

`Fascia` dovrebbe coincidere con la prima lettera di `Zona` (B1 → B).

In [19]:
(df["Fascia"] != df["Zona"].str[0]).sum()

np.int64(0)

0 incoerenze.

## 4.7 Stato, superficie e copertura

In [20]:
pd.crosstab(df["Stato"], df["Stato_prev"])

Stato_prev,,P
Stato,,
NORMALE,723,141146
OTTIMO,12482,7244
SCADENTE,201,385


La tabella incrocia `Stato` (righe) con `Stato_prev` (colonne). Le righe con «P» sono 148.787; le 13.406 senza «P» sono quasi tutte OTTIMO (12.482). Confronto con le ripetizioni di 4.2:

In [21]:
prevalente = df["Stato_prev"].str.strip() == "P"

print("Righe con chiave ripetuta (4.2):", df.duplicated(subset=CHIAVE).sum())
print("Righe senza P in Stato_prev:    ", (~prevalente).sum())
print()
print("Quante righe P per ogni coppia zona + tipologia:")
print(prevalente.groupby([df["LinkZona"], df["Cod_Tip"]]).sum().value_counts())

Righe con chiave ripetuta (4.2): 13406
Righe senza P in Stato_prev:     13406

Quante righe P per ogni coppia zona + tipologia:
Stato_prev
1    148787
Name: count, dtype: int64


Le due cifre coincidono (13.406): le righe in più sono esattamente quelle degli stati non prevalenti, e ogni coppia zona + tipologia ha la sua riga «P».

Copertura geografica:

In [22]:
print("Regioni:", df["Regione"].nunique())
print("Province:", df["Prov"].nunique())
print("Comuni:", df["Comune_ISTAT"].nunique())
print("Righe senza provincia:", df["Prov"].isna().sum())

Regioni: 20
Province: 103
Comuni: 8004
Righe senza provincia: 0


Il semestre copre 20 regioni, 103 sigle di provincia e 8.004 comuni. Nessuna riga è senza provincia: «NA» di Napoli è letta correttamente.

## Esiti dei controlli

- **Mancanti:** affitto 8,5%, `Sup_NL_loc` 8,4%; acquisto 13 righe su 162.193; `Stato` 12 righe; `Comune_ISTAT` 1 riga.
- **Duplicati:** nessuna riga identica. 13.406 righe ripetono zona + tipologia perché c'è una riga per stato di conservazione; quella dello stato prevalente ha `Stato_prev` = P.
- **Codici:** `Cod_Tip` e `LinkZona` sono corrispondenze uno a uno.
- **Zeri:** 215 prezzi di acquisto e 373 di affitto, da trattare come mancanti.
- **Anomali:** nessun negativo o minimo > massimo. I massimi sono plausibili e restano.
- **Altro:** `Fascia` coincide sempre con `Zona`. L'acquisto è sempre su superficie lorda, l'affitto in circa un quarto dei casi su netta.

# Fase 5 · Decisioni e tabella pulita (un semestre)

**Decisione 1 · Quali tipologie.** Il progetto confronta tre segmenti:
- **Abitazioni:** civili, di tipo economico, ville e villini, signorili, tipiche dei luoghi
- **Negozi**
- **Uffici:** uffici e uffici strutturati

Le altre (box, posti auto, autorimesse, magazzini, laboratori, capannoni, centri commerciali, pensioni) non rientrano nell'obiettivo e alcune non sono confrontabili con il resto.

In [23]:
df["Descr_Tipologia"].value_counts()

Descr_Tipologia
Abitazioni civili                26334
Ville e Villini                  18540
Abitazioni di tipo economico     17145
Negozi                           16793
Box                              16684
Magazzini                        14282
Uffici                           12398
Laboratori                       12122
Capannoni tipici                  9300
Capannoni industriali             5827
Autorimesse                       4344
Posti auto scoperti               3371
Posti auto coperti                3217
Centri commerciali                 708
Uffici strutturati                 428
Abitazioni tipiche dei luoghi      352
Abitazioni signorili               217
Pensioni e assimilati              131
Name: count, dtype: int64

In [24]:
SEGMENTI = {
    "Abitazioni civili": "Abitazioni",
    "Abitazioni di tipo economico": "Abitazioni",
    "Ville e Villini": "Abitazioni",
    "Abitazioni signorili": "Abitazioni",
    "Abitazioni tipiche dei luoghi": "Abitazioni",
    "Negozi": "Negozi",
    "Uffici": "Uffici",
    "Uffici strutturati": "Uffici",
}

**Decisione 2 · Gli zeri.** Un prezzo di 0 €/m² non è credibile: diventa dato mancante, senza cancellare la riga.

In [25]:
df_pulito = df.copy()
df_pulito[PREZZI] = df_pulito[PREZZI].where(df_pulito[PREZZI] > 0)
df_pulito["Segmento"] = df_pulito["Descr_Tipologia"].map(SEGMENTI)

print("Righe prima:", len(df_pulito))
df_pulito = df_pulito[df_pulito["Segmento"].notna()].copy()
print("Righe dopo:", len(df_pulito))
df_pulito["Segmento"].value_counts()

Righe prima: 162193
Righe dopo: 92207


Segmento
Abitazioni    62588
Negozi        16793
Uffici        12826
Name: count, dtype: int64

Dalle 162.193 righe del semestre restano 92.207: 62.588 abitazioni, 16.793 negozi, 12.826 uffici.

**Decisione 3 · Righe senza prezzo di acquisto.** Restano, con il vuoto. Media, mediana e conteggio ignorano i vuoti.

# Fase 6 · Tutti i semestri

## 6.1 Una funzione per leggere un file

Per ripetere gli stessi passi su 20 file serve una **funzione**: un blocco di istruzioni con un nome, che riceve un file e restituisce la tabella pulita (si definisce con `def`).

Il semestre si legge dalla prima riga del file («Semestre 2016/1») e non dal nome, perché i nomi dei file si sono rivelati inaffidabili (vedi Fase 8). Titolo e nome vengono confrontati.

In [26]:
COLONNE = ["Area_territoriale", "Regione", "Prov", "Comune_ISTAT", "Comune_descrizione",
           "Fascia", "Zona", "LinkZona", "Cod_Tip", "Descr_Tipologia", "Stato", "Stato_prev",
           "Compr_min", "Compr_max", "Sup_NL_compr", "Loc_min", "Loc_max", "Sup_NL_loc"]


def leggi_semestre(nome_file):
    percorso = os.path.join(CARTELLA_OMI, nome_file)
    with open(percorso, encoding="utf-8") as f:
        titolo = f.readline()
    trovato = re.search(r"Semestre (\d{4})/(\d)", titolo)
    if trovato is None:
        raise ValueError("Semestre non trovato nel titolo di " + nome_file)
    anno = int(trovato.group(1))
    sem = int(trovato.group(2))

    dati = pd.read_csv(percorso, sep=";", skiprows=1, decimal=",",
                       keep_default_na=False, na_values=[""],
                       usecols=COLONNE, dtype={"Comune_ISTAT": str})
    dati[PREZZI] = dati[PREZZI].where(dati[PREZZI] > 0)
    dati["Segmento"] = dati["Descr_Tipologia"].map(SEGMENTI)
    dati = dati[dati["Segmento"].notna()].copy()

    dati["Anno"] = anno
    dati["Sem"] = sem
    dati["Semestre"] = str(anno) + "-H" + str(sem)
    dati["File"] = nome_file
    return dati

## 6.2 Caricamento dei 20 semestri

Il ciclo legge i file uno alla volta e stampa semestre e numero di righe. Se nome e titolo non coincidono compare un avviso.

In [27]:
tabelle = []
for nome in file_valori:
    t = leggi_semestre(nome)
    tabelle.append(t)

    dal_nome = nome.split("_")[3]
    dal_titolo = str(t["Anno"].iloc[0]) + str(t["Sem"].iloc[0])
    avviso = "" if dal_nome == dal_titolo else "  <-- ATTENZIONE: il nome dice " + dal_nome
    print(nome, "->", t["Semestre"].iloc[0], len(t), "righe" + avviso)

omi = pd.concat(tabelle, ignore_index=True)
omi.shape

QI_1444537_1_20161_VALORI.csv -> 2016-H1 92207 righe
QI_1444538_1_20162_VALORI.csv -> 2016-H2 92267 righe
QI_1444540_1_20171_VALORI.csv -> 2017-H1 92218 righe
QI_1444541_1_20172_VALORI.csv -> 2017-H2 92270 righe
QI_1444543_1_20182_VALORI.csv -> 2018-H2 92342 righe
QI_1444544_1_20191_VALORI.csv -> 2019-H1 91786 righe
QI_1444545_1_20192_VALORI.csv -> 2019-H2 91786 righe
QI_1444546_1_20201_VALORI.csv -> 2020-H1 91819 righe
QI_1444549_1_20211_VALORI.csv -> 2021-H1 92210 righe
QI_1444550_1_20212_VALORI.csv -> 2021-H2 92320 righe
QI_1444551_1_20221_VALORI.csv -> 2022-H1 92245 righe
QI_1444552_1_20222_VALORI.csv -> 2022-H2 92204 righe
QI_1444553_1_20231_VALORI.csv -> 2023-H1 92256 righe
QI_1444554_1_20232_VALORI.csv -> 2023-H2 92299 righe
QI_1444555_1_20241_VALORI.csv -> 2024-H1 92186 righe
QI_1444556_1_20242_VALORI.csv -> 2024-H2 91149 righe
QI_1444557_1_20251_VALORI.csv -> 2025-H1 91153 righe
QI_1444558_1_20252_VALORI.csv -> 2025-H2 91164 righe
QI_1451633_1_20181_VALORI.csv -> 2018-H1 92283

(1839995, 23)

I 20 file danno 20 semestri distinti, da 2016-H1 a 2025-H2, senza avvisi. La tabella unita ha 1.839.995 righe e 23 colonne. Il caricamento è pesante e richiede alcuni minuti.

In [28]:
omi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1839995 entries, 0 to 1839994
Data columns (total 23 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Area_territoriale   object 
 1   Regione             object 
 2   Prov                object 
 3   Comune_ISTAT        object 
 4   Comune_descrizione  object 
 5   Fascia              object 
 6   Zona                object 
 7   LinkZona            object 
 8   Cod_Tip             int64  
 9   Descr_Tipologia     object 
 10  Stato               object 
 11  Stato_prev          object 
 12  Compr_min           float64
 13  Compr_max           float64
 14  Sup_NL_compr        object 
 15  Loc_min             float64
 16  Loc_max             float64
 17  Sup_NL_loc          object 
 18  Segmento            object 
 19  Anno                int64  
 20  Sem                 int64  
 21  Semestre            object 
 22  File                object 
dtypes: float64(4), int64(3), object(16)
memory usage: 322.9+ 

In [29]:
print(omi["Segmento"].value_counts())
print(omi["Segmento"].isna().sum())

Segmento
Abitazioni    1258098
Negozi         335011
Uffici         246886
Name: count, dtype: int64
0


## 6.3 Completezza dei semestri

Ci aspettiamo 20 semestri, ognuno da un solo file.

In [30]:
attesi = [str(a) + "-H" + str(s) for a in range(2016, 2026) for s in (1, 2)]
presenti = list(omi["Semestre"].unique())
print("Semestri attesi ma assenti:", [s for s in attesi if s not in presenti])
print("Semestri presenti piu' di una volta:", list(omi.groupby("Semestre")["File"].nunique()[lambda x: x > 1].index))

Semestri attesi ma assenti: []
Semestri presenti piu' di una volta: []


Nessun semestre assente o ripetuto.

Righe, zone e comuni per semestre, con la variazione % delle righe sul semestre precedente:

In [31]:
riepilogo = omi.groupby("Semestre").agg(righe=("LinkZona", "size"),
                                         zone=("LinkZona", "nunique"),
                                         comuni=("Comune_ISTAT", "nunique"))
riepilogo["variazione_righe"] = riepilogo["righe"].pct_change().round(3)
riepilogo

,righe,zone,comuni,variazione_righe
Semestre,,,,
2016-H1,92207,23627,8004,NaN
2016-H2,92267,23625,7998,0.001
2017-H1,92218,23663,7987,-0.001
2017-H2,92270,23668,7986,0.001
2018-H1,92283,23673,7969,0.000
2018-H2,92342,23674,7964,0.001
2019-H1,91786,23630,7940,-0.006
2019-H2,91786,23630,7930,0.000
2020-H1,91819,23631,7927,0.000


- Le righe sono stabili (91.000-92.400 per semestre), con due cali: 2019-H1 (−0,6%) e 2024-H2 (−1,1%).
- I comuni scendono da 8.004 a 7.888 per le fusioni, non per errori. Il calo del 2019-H1 coincide con −24 comuni e −44 zone.
- Nel 2024-H2 le zone passano da 23.616 a 23.375 (−241) mentre i comuni restano quasi uguali (7.891 → 7.889). Potrebbe essere una ridefinizione delle zone, ma non è verificato.
- I confronti nel tempo si fanno quindi su aggregati (regione, ripartizione, segmento), non zona per zona.

## 6.4 Prime letture dei prezzi

Prezzi medi per semestre su tutte le righe. È solo un controllo di coerenza: la media è semplice e mescola tipologie, zone e stati.

In [32]:
omi.groupby("Semestre")[PREZZI].mean().round(2)

,Compr_min,Compr_max,Loc_min,Loc_max
Semestre,,,,
2016-H1,1044.36,1407.76,3.91,5.38
2016-H2,1041.03,1404.39,3.91,5.39
2017-H1,1034.35,1396.59,3.90,5.37
2017-H2,1026.70,1387.95,3.88,5.35
2018-H1,1021.95,1383.30,3.87,5.35
2018-H2,1016.18,1377.48,3.86,5.34
2019-H1,1014.10,1376.24,3.86,5.35
2019-H2,1009.52,1372.93,3.86,5.36
2020-H1,1001.98,1364.32,3.85,5.36


- I prezzi di acquisto scendono fino al 2022-H1 (`Compr_min` da 1.044 a 984 €/m², circa −6%) e poi risalgono fino a 1.000 €/m² nel 2025-H2. Gli affitti hanno la stessa forma, con risalita più marcata (`Loc_min` da 3,79 a 3,99 €/m², circa +5%).
- 2019-H1 e 2019-H2 hanno le stesse righe e zone ma prezzi diversi (`Compr_min` 1.014 contro 1.010): sono due file distinti.

## 6.5 Mancanti e valori impossibili sul totale

In [33]:
print("Prezzi mancanti:")
print(omi[PREZZI].isna().sum())
print()
print("Acquisto con minimo > massimo:", (omi["Compr_min"] > omi["Compr_max"]).sum())
print("Affitto con minimo > massimo: ", (omi["Loc_min"] > omi["Loc_max"]).sum())

Prezzi mancanti:
Compr_min     10802
Compr_max     10802
Loc_min      195511
Loc_max      195511
dtype: int64

Acquisto con minimo > massimo: 0
Affitto con minimo > massimo:  7


I prezzi di acquisto mancano in 10.802 righe (0,6%), quelli di affitto in 195.511 (10,6%). Nessun acquisto ha minimo > massimo; 7 affitti sì (Decisione 5, in 6.7).

Quota di `Loc_min` mancante per semestre:

In [34]:
omi["Loc_min"].isna().groupby(omi["Semestre"]).mean().round(3)

Semestre
2016-H1    0.098
2016-H2    0.115
2017-H1    0.110
2017-H2    0.106
2018-H1    0.106
2018-H2    0.104
2019-H1    0.105
2019-H2    0.104
2020-H1    0.105
2020-H2    0.104
2021-H1    0.108
2021-H2    0.107
2022-H1    0.106
2022-H2    0.106
2023-H1    0.106
2023-H2    0.106
2024-H1    0.106
2024-H2    0.107
2025-H1    0.107
2025-H2    0.107
Name: Loc_min, dtype: float64

Dal 2017 la quota resta tra il 10,4% e l'11,0% (2016: 9,8% in H1, 11,5% in H2). I mancanti sono stabili nel tempo e non distorcono il confronto tra periodi.

## 6.6 Valori estremi sul totale

In [35]:
omi[PREZZI].describe().round(2)

,Compr_min,Compr_max,Loc_min,Loc_max
count,1829193.00,1829193.00,1644484.00,1644484.00
mean,1004.69,1368.49,3.87,5.39
std,657.30,921.22,2.46,3.62
min,60.00,120.00,0.40,0.50
25%,600.00,830.00,2.30,3.20
50%,850.00,1150.00,3.30,4.50
75%,1200.00,1600.00,4.70,6.60
max,16600.00,30000.00,95.00,180.00


`count` conta solo i valori presenti, quindi le basi sono diverse: 1.829.193 righe per i prezzi di acquisto, 1.644.484 per gli affitti. Affitto medio × 12 (3,87 × 12) diviso prezzo medio (1.005 €/m²) dà circa il 4,6%: un rendimento realistico, che conferma l'affitto mensile.

I valori più alti, per affitto e per acquisto:

In [36]:
omi.nlargest(5, "Loc_max")[["Comune_descrizione", "Zona", "Descr_Tipologia", "Semestre", "Loc_max"]]

,Comune_descrizione,Zona,Descr_Tipologia,Semestre,Loc_max
1473050,ARZACHENA,E7,Negozi,2024-H2,180.0
1490504,MILANO,B12,Negozi,2025-H1,180.0
1564203,ARZACHENA,E7,Negozi,2025-H1,180.0
1581656,MILANO,B12,Negozi,2025-H2,180.0
1655367,ARZACHENA,E7,Negozi,2025-H2,180.0


In [37]:
omi.nlargest(5, "Compr_max")[["Comune_descrizione", "Zona", "Descr_Tipologia", "Semestre", "Compr_max"]]

,Comune_descrizione,Zona,Descr_Tipologia,Semestre,Compr_max
1473050,ARZACHENA,E7,Negozi,2024-H2,30000.0
1564203,ARZACHENA,E7,Negozi,2025-H1,30000.0
1655367,ARZACHENA,E7,Negozi,2025-H2,30000.0
1399350,MILANO,B12,Negozi,2024-H2,29000.0
1490504,MILANO,B12,Negozi,2025-H1,29000.0


Prezzi di acquisto più alti tra le sole abitazioni:

In [38]:
abitazioni = omi[omi["Segmento"] == "Abitazioni"]
abitazioni.nlargest(5, "Compr_max")[["Comune_descrizione", "Zona", "Descr_Tipologia", "Semestre", "Compr_max"]]

,Comune_descrizione,Zona,Descr_Tipologia,Semestre,Compr_max
25808,PORTOFINO,B1,Abitazioni signorili,2016-H1,22900.0
118010,PORTOFINO,B1,Abitazioni signorili,2016-H2,22900.0
210395,PORTOFINO,B1,Abitazioni signorili,2017-H1,22900.0
302634,PORTOFINO,B1,Abitazioni signorili,2017-H2,22900.0
394899,PORTOFINO,B1,Abitazioni signorili,2018-H2,22900.0


**Decisione 6 · I valori estremi restano.** I massimi sono negozi ad Arzachena (Costa Smeralda) e Milano, e abitazioni signorili a Portofino (22.900 €/m² in tutti i semestri mostrati, dal 2016-H1 al 2018-H2). Sono realistici per quei mercati e fanno parte del fenomeno studiato: eliminarli distorcerebbe il quadro.

Il valore di Portofino, invariato per anni, mostra che le quotazioni OMI cambiano «a gradini»: è un limite della fonte (Fase 8).

## 6.7 Una riga per zona × tipologia × semestre

Quante righe ripetono la stessa chiave nel totale?

In [39]:
CHIAVE_TOT = ["Semestre", "LinkZona", "Cod_Tip"]
print("Righe con chiave ripetuta nel totale:", omi.duplicated(subset=CHIAVE_TOT).sum())

Righe con chiave ripetuta nel totale: 230027


Sono 230.027, oltre il 12% delle righe: quelle degli stati non prevalenti (vedi 4.2 e 4.7). Prima di contarle si ripuliscono `Stato_prev` e `Sup_NL_loc` (396 celle con solo spazi nel semestre di prova): gli spazi diventano vuoti veri.

In [40]:
omi["Stato_prev"] = omi["Stato_prev"].str.strip()
sup = omi["Sup_NL_loc"].str.strip()
omi["Sup_NL_loc"] = sup.mask(sup == "")

print("Righe con chiave ripetuta:   ", omi.duplicated(subset=CHIAVE_TOT).sum())
print("Righe con Stato_prev diverso da P:", (omi["Stato_prev"] != "P").sum())

Righe con chiave ripetuta:    230027
Righe con Stato_prev diverso da P: 230029


**Decisione 4 · Una riga per zona × tipologia × semestre: lo stato prevalente.** Tenere tutti gli stati mescola immobili in condizioni diverse (le righe OTTIMO costano di più) e dà più peso alle zone con più stati pubblicati. Si tiene quindi la riga con `Stato_prev` = P. Se una coppia non avesse nessuna riga P, si terrebbe la prima disponibile; il notebook conta quante volte succede.

In [41]:
righe_prima = len(omi)

omi["_prevalente"] = omi["Stato_prev"] == "P"
omi = omi.sort_values("_prevalente", ascending=False, kind="stable")
omi = omi.drop_duplicates(subset=CHIAVE_TOT, keep="first").sort_index()

senza_prevalente = (~omi["_prevalente"]).sum()
omi = omi.drop(columns="_prevalente").reset_index(drop=True)

print("Righe prima:", righe_prima)
print("Righe dopo: ", len(omi))
print("Coppie senza riga P (tenuta la prima disponibile):", senza_prevalente)
print("Righe con chiave ripetuta dopo la pulizia:", omi.duplicated(subset=CHIAVE_TOT).sum())

Righe prima: 1839995
Righe dopo:  1609968
Coppie senza riga P (tenuta la prima disponibile): 2
Righe con chiave ripetuta dopo la pulizia: 0


Ora la chiave semestre + zona + tipologia è unica.

**Decisione 5 · Affitti con minimo > massimo.** Un intervallo invertito non è credibile: i due valori di affitto della riga diventano mancanti. Il prezzo di acquisto resta.

In [42]:
affitto_invertito = omi["Loc_min"] > omi["Loc_max"]
print("Affitti con minimo > massimo (resi mancanti):", affitto_invertito.sum())
omi.loc[affitto_invertito, ["Loc_min", "Loc_max"]] = float("nan")
print("Controllo dopo la correzione:", (omi["Loc_min"] > omi["Loc_max"]).sum())

Affitti con minimo > massimo (resi mancanti): 1
Controllo dopo la correzione: 0


Quota di prezzi mancanti nella tabella finale (%):

In [43]:
print(omi.shape)
(omi[PREZZI].isna().mean() * 100).round(1)

(1609968, 23)


Compr_min    0.5
Compr_max    0.5
Loc_min      9.2
Loc_max      9.2
dtype: float64

Nella tabella finale i prezzi di acquisto mancano in meno dell'1% delle righe, quelli di affitto in circa il 10%. Le righe senza prezzo restano (Decisione 3).

# Fase 7 · Collegare le fonti

## 7.1 Nomi dei luoghi

**Decisione 7.** In alcuni nomi l'apostrofo è un apice rovesciato (``CORTINA D`AMPEZZO``) e la Valle d'Aosta ha un nome bilingue con lo stesso carattere. Si sostituisce l'apice con l'apostrofo e si usa il nome italiano, per avere nomi uniformi nei grafici e nei collegamenti tra tabelle.

In [44]:
for colonna in ["Regione", "Comune_descrizione"]:
    omi[colonna] = omi[colonna].str.replace("`", "'", regex=False)
omi["Regione"] = omi["Regione"].replace({"VALLE D'AOSTA/VALLE'E D'AOSTE": "VALLE D'AOSTA"})

print("Regioni:", omi["Regione"].nunique())
print(sorted(omi["Regione"].unique()))

Regioni: 20
['ABRUZZO', 'BASILICATA', 'CALABRIA', 'CAMPANIA', 'EMILIA-ROMAGNA', 'FRIULI-VENEZIA GIULIA', 'LAZIO', 'LIGURIA', 'LOMBARDIA', 'MARCHE', 'MOLISE', 'PIEMONTE', 'PUGLIA', 'SARDEGNA', 'SICILIA', 'TOSCANA', 'TRENTINO-ALTO ADIGE', 'UMBRIA', "VALLE D'AOSTA", 'VENETO']


20 regioni con nomi puliti.

## 7.2 Ripartizione geografica

L'OMI ha 5 aree (Nord-Ovest, Nord-Est, Centro, Sud, Isole). Il progetto ne usa 3: **Nord**, **Centro**, **Sud e Isole**, come nella suddivisione Istat del Mezzogiorno.

**Decisione 8.** Prima di applicare la mappa si guardano le regioni di ogni area OMI:

In [45]:
for area, regioni in omi.groupby("Area_territoriale")["Regione"].unique().items():
    print(area, "->", list(regioni))

CENTRO -> ['TOSCANA', 'UMBRIA', 'MARCHE', 'LAZIO']
ISOLE -> ['SICILIA', 'SARDEGNA']
NORD-EST -> ['TRENTINO-ALTO ADIGE', 'VENETO', 'FRIULI-VENEZIA GIULIA', 'EMILIA-ROMAGNA']
NORD-OVEST -> ['PIEMONTE', "VALLE D'AOSTA", 'LOMBARDIA', 'LIGURIA']
SUD -> ['ABRUZZO', 'MOLISE', 'CAMPANIA', 'PUGLIA', 'BASILICATA', 'CALABRIA']


Nord-Ovest e Nord-Est contengono le 8 regioni del Nord (Piemonte, Valle d'Aosta, Lombardia, Liguria, Trentino-Alto Adige, Veneto, Friuli-Venezia Giulia, Emilia-Romagna); Centro ha Toscana, Umbria, Marche, Lazio; Sud ha Abruzzo, Molise, Campania, Puglia, Basilicata, Calabria; Isole ha Sicilia e Sardegna. Corrisponde alla ripartizione Istat, quindi la mappa si applica senza correzioni.

In [46]:
AREA_MAP = {
    "NORD-OVEST": "Nord",
    "NORD-EST": "Nord",
    "CENTRO": "Centro",
    "SUD": "Sud e Isole",
    "ISOLE": "Sud e Isole",
}
omi["Ripartizione"] = omi["Area_territoriale"].map(AREA_MAP)

print("Righe senza ripartizione:", omi["Ripartizione"].isna().sum())
print(omi[omi["Ripartizione"].isna()]["Area_territoriale"].value_counts())
print()
print("Napoli (NA):")
print(omi[omi["Prov"] == "NA"]["Ripartizione"].value_counts())
print()
print("Regioni per ripartizione:")
print(omi.groupby("Ripartizione")["Regione"].nunique())

Righe senza ripartizione: 0
Series([], Name: count, dtype: int64)

Napoli (NA):
Ripartizione
Sud e Isole    30296
Name: count, dtype: int64

Regioni per ripartizione:
Ripartizione
Centro         4
Nord           8
Sud e Isole    8
Name: Regione, dtype: int64


Nessuna riga resta senza ripartizione, Napoli («NA») è tutta in «Sud e Isole» e le regioni sono 8 (Nord), 4 (Centro), 8 (Sud e Isole), 20 in tutto.

## 7.3 Regola per le date

L'Istat ha dati **trimestrali**, l'OMI **semestrali**.

**Decisione 9.** H1 copre gennaio-giugno (trimestri Q1 e Q2), H2 luglio-dicembre (Q3 e Q4). Nel collegamento, i trimestri Istat si associano al semestre OMI che li contiene. La colonna `Data` è solo un'etichetta, il primo giorno del semestre, usata per ordinare e per l'asse del tempo.

In [47]:
mese = omi["Sem"].map({1: "01", 2: "07"})
omi["Data"] = pd.to_datetime(omi["Anno"].astype(str) + "-" + mese + "-01")
omi[["Semestre", "Data"]].drop_duplicates().sort_values("Data").reset_index(drop=True)

,Semestre,Data
0,2016-H1,2016-01-01
1,2016-H2,2016-07-01
2,2017-H1,2017-01-01
3,2017-H2,2017-07-01
4,2018-H1,2018-01-01
5,2018-H2,2018-07-01
6,2019-H1,2019-01-01
7,2019-H2,2019-07-01
8,2020-H1,2020-01-01
9,2020-H2,2020-07-01


20 etichette, dal 1 gennaio 2016 (2016-H1) al 1 luglio 2025 (2025-H2), una ogni sei mesi.

# Fase 8 · Salvataggio e documentazione

In [48]:
os.makedirs(CARTELLA_OUT, exist_ok=True)
percorso_out = os.path.join(CARTELLA_OUT, "omi_semestri_puliti.csv")
omi.to_csv(percorso_out, index=False)
print("Salvato:", percorso_out, omi.shape)

Salvato: ../data/processed\omi_semestri_puliti.csv (1609968, 25)


In [49]:
dimensione_mb = os.path.getsize(percorso_out) / 1_000_000
print("Dimensione del file:", round(dimensione_mb), "MB")
if dimensione_mb > 100:
    print("Supera i 100 MB di GitHub: il file non va versionato (va escluso con .gitignore).")

Dimensione del file: 296 MB
Supera i 100 MB di GitHub: il file non va versionato (va escluso con .gitignore).


Il file salvato viene riletto per controllare che righe, colonne, prezzi mancanti e righe di Napoli coincidano con la tabella in memoria. La rilettura richiede le stesse opzioni della lettura iniziale: senza `keep_default_na=False`, pandas legge «NA» come dato mancante e Napoli perde la provincia.

In [50]:
prova = pd.read_csv(percorso_out, keep_default_na=False, na_values=[""],
                    dtype={"Comune_ISTAT": str}, parse_dates=["Data"])

print("Righe e colonne rilette:", prova.shape, "- in memoria:", omi.shape)
print("Prezzi mancanti:", int(prova[PREZZI].isna().sum().sum()), "- in memoria:", int(omi[PREZZI].isna().sum().sum()))
print("Righe di Napoli (Prov = NA):", (prova["Prov"] == "NA").sum())
print("Tipo della colonna Data:", prova["Data"].dtype)

Righe e colonne rilette: (1609968, 25) - in memoria: (1609968, 25)
Prezzi mancanti: 314054 - in memoria: 314054
Righe di Napoli (Prov = NA): 30296
Tipo della colonna Data: datetime64[ns]


**GitHub.** Il CSV pulito pesa centinaia di MB e GitHub rifiuta i file sopra i 100 MB. `data/processed/omi_semestri_puliti.csv` e `data/Raw/Omi/` non sono quindi nel repository (regole nel `.gitignore`): il primo si rigenera con questo notebook, i secondi si scaricano dall'ente. Nel repository ci sono i notebook, i documenti e le tabelle aggregate più piccole dell'analisi.

## Decisioni di pulizia

| N. | Decisione | Motivo |
|---|---|---|
| 1 | Si tengono abitazioni (5 tipologie), negozi, uffici (2 tipologie) | Sono i segmenti del progetto |
| 2 | Prezzi uguali a zero → mancanti | 0 €/m² non è un prezzo credibile |
| 3 | Righe senza prezzo di acquisto → restano | Pandas ignora i vuoti nei calcoli |
| 4 | Una riga per semestre + zona + tipologia: lo stato prevalente (`Stato_prev` = P) | Non mescola stati diversi e non dà più peso alle zone con più stati |
| 5 | Affitto con minimo > massimo → mancante | Intervallo invertito non credibile |
| 6 | Valori estremi mantenuti | Negozi e ville in località di pregio, realistici |
| 7 | Apice rovesciato → apostrofo; «Valle d'Aosta» | Nomi uniformi |
| 8 | 5 aree OMI → Nord, Centro, Sud e Isole | Come la suddivisione Istat |
| 9 | `Data` = primo giorno del semestre; H1 ↔ Q1+Q2, H2 ↔ Q3+Q4 | Allineare OMI e Istat |

## Limiti dei dati

| Problema | Cosa è successo | Come è stato gestito |
|---|---|---|
| Intestazione con titolo | `read_csv` vedeva 1 colonna | `skiprows=1`, `sep=";"` |
| Decimali con virgola | Prezzi letti come testo | `decimal=","` |
| Sigla «NA» di Napoli | Letta come dato mancante (2.660 righe nel 2016-H1) | `keep_default_na=False`, `na_values=[""]` |
| Colonna vuota | `;` finale di ogni riga | Eliminata con `usecols` |
| Spazi al posto dei vuoti | `Stato_prev` e `Sup_NL_loc` | `str.strip()` in 6.7 |
| Prezzi a zero | Non credibili | Trasformati in mancanti |
| Nomi dei file inaffidabili | Un semestre compariva due volte, altri due non si trovavano | File riscaricati; il semestre è letto dal titolo |
| Più righe per zona e tipologia | Una per stato di conservazione | Stato prevalente (Decisione 4) |
| Superficie netta o lorda | Affitto su netta in circa un quarto dei casi | `Sup_NL_loc` resta nella tabella per distinguerli |
| Affitti mancanti | Circa il 10%, stabile nel tempo | Restano vuoti |
| Quotazioni «a gradini» | Stesso valore per più semestri (es. Portofino) | Stime di larga massima: variazioni piccole da leggere con cautela |
| Comuni e zone che cambiano | Fusioni (da 8.004 a 7.888 comuni) e zone ridotte nel 2024-H2 (−241) | Confronti su aggregati |
| Apice nei nomi | ``D`AMPEZZO``, Valle d'Aosta bilingue | Sostituiti (Decisione 7) |
| Dimensione del file | Oltre il limite di GitHub | Fuori dal repository, rigenerabile |

## Come si riproduce
1. Scaricare i 20 file `QI_..._VALORI.csv` dal servizio *Forniture dati OMI* in `data/Raw/Omi/`.
2. Eseguire il notebook con `Restart Kernel and Run All Cells`.
3. Il risultato è `data/processed/omi_semestri_puliti.csv`, usato dai notebook di analisi e dall'esportazione per Power BI.

**Fonte:** «Agenzia delle Entrate – OMI», licenza CC-BY 4.0.